# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_14 — Fixed Income Duration, Convexity, and Curve Risk

### Research question

How do we measure, attribute, hedge, and stress-test fixed-income portfolio risk using yield curves, duration, convexity, DV01/PV01, key-rate duration, and curve-shock scenarios?

This notebook closes Phase 4:

```text
04_01_multi_asset_return_engine
04_02_capm_and_performance_attribution
04_03_volatility_targeting_and_position_sizing
04_04_mean_variance_optimization_ledoit_wolf
04_05_pca_spectral_risk_analysis
04_06_var_cvar_and_stress_testing
04_07_beta_weighted_tail_risk_hedging
04_08_futures_minimum_variance_hedge_ratio
04_09_risk_parity_and_equal_risk_contribution
04_10_hierarchical_risk_parity
04_11_cvar_convex_optimization
04_12_black_litterman_allocation
04_13_stochastic_control_portfolio_problem
04_14_fixed_income_duration_convexity_risk
```

It covers:

1. zero curves and discount factors;
2. coupon bond cashflows;
3. clean price from a curve;
4. yield to maturity;
5. Macaulay duration;
6. modified duration;
7. effective duration;
8. convexity;
9. DV01 / PV01;
10. key-rate duration;
11. portfolio duration and convexity;
12. curve scenario shocks;
13. duration-convexity approximation error;
14. barbell versus bullet portfolios;
15. immunisation of liabilities;
16. futures-style duration hedging;
17. rolling yield-curve risk simulation;
18. factor risk from level, slope, and curvature;
19. stress testing and governance checks;
20. saved outputs and manifest.

The key idea:

> Fixed-income risk is not just “bond prices fall when yields rise.” It is curve location, cashflow timing, duration, convexity, spread, key-rate exposure, and hedge basis risk.

## 1. Bond pricing from a zero curve

For cashflows $CF_i$ paid at times $t_i$, using continuously compounded zero rates $r(t_i)$, price is:

$$
P = \sum_i CF_i e^{-r(t_i)t_i}
$$

If a bond has a spread $s$, a simple spread-adjusted discounting approximation is:

$$
P = \sum_i CF_i e^{-(r(t_i)+s)t_i}
$$

This notebook uses this curve-discounting approach for pricing and risk.

## 2. Duration intuition

Duration measures sensitivity of bond price to yield changes.

### Macaulay duration

$$
D_{Mac} = \sum_i t_i \frac{PV(CF_i)}{P}
$$

It is a present-value-weighted average cashflow time.

### Modified duration

$$
D_{Mod} = \frac{D_{Mac}}{1+y/m}
$$

where $y$ is yield to maturity and $m$ is coupon frequency.

Approximate price change:

$$
\frac{\Delta P}{P} \approx -D_{Mod}\Delta y
$$

Duration is a first-order approximation.

## 3. Convexity intuition

Bond price-yield relationship is curved.

Convexity captures second-order sensitivity:

$$
\frac{\Delta P}{P} \approx -D\Delta y + \frac{1}{2}C(\Delta y)^2
$$

For larger yield moves, convexity improves the approximation.

Positive convexity means price gains from a yield fall are larger than price losses from an equal yield rise, all else equal.

## 4. DV01 / PV01

DV01 measures dollar change in price for a 1 basis point yield move.

For portfolio value $V$ and modified duration $D$:

$$
DV01 \approx V \times D \times 0.0001
$$

For a long bond position:

- yields up 1 bp → price falls by roughly DV01;
- yields down 1 bp → price rises by roughly DV01.

DV01 is the core language of fixed-income risk desks.

## 5. Key-rate duration

A parallel duration assumes every point on the curve moves together.

But curves twist.

Key-rate duration measures sensitivity to specific maturities:

$$
KRD_k = -\frac{P(r_k+\Delta)-P(r_k-\Delta)} {2P\Delta}
$$

where only one curve knot $k$ is bumped.

This identifies whether a portfolio is exposed to 2-year, 5-year, 10-year, or 30-year curve moves.

## 6. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class FixedIncomeRiskConfig:
    n_days: int = 1_500
    annualisation: int = 252
    seed: int = 42
    face_value: float = 100.0
    coupon_frequency: int = 2
    bump_bp: float = 1.0
    shock_bp: float = 100.0
    initial_capital: float = 10_000_000.0
    transaction_cost_bp_notional: float = 0.5
    futures_price: float = 112.0
    futures_multiplier: float = 1000.0


config = FixedIncomeRiskConfig()
config

## 7. Simulate a yield curve history

We create synthetic zero curves with:

1. level factor;
2. slope factor;
3. curvature factor;
4. inflation shock episodes;
5. recession shock episodes;
6. curve twists.

This gives a realistic fixed-income risk playground.

In [ ]:
def simulate_yield_curve_history(config: FixedIncomeRiskConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2018-01-01", periods=config.n_days)

    tenors = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30], dtype=float)

    base_curve = pd.Series(
        [0.020, 0.021, 0.022, 0.024, 0.026, 0.029, 0.031, 0.033, 0.036, 0.038],
        index=tenors,
        name="base_zero_rate"
    )

    level = np.zeros(config.n_days)
    slope = np.zeros(config.n_days)
    curvature = np.zeros(config.n_days)
    regime = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(1, config.n_days):
        level[t] = 0.995 * level[t-1] + 0.00045 * rng.standard_normal()
        slope[t] = 0.990 * slope[t-1] + 0.00065 * rng.standard_normal()
        curvature[t] = 0.985 * curvature[t-1] + 0.00075 * rng.standard_normal()

        u = rng.uniform()
        if u < 0.006:
            regime[t] = "inflation_shock"
            level[t] += rng.uniform(0.006, 0.018)
            slope[t] += rng.uniform(0.002, 0.010)
            curvature[t] += rng.normal(0.000, 0.006)
        elif u < 0.011:
            regime[t] = "recession_rally"
            level[t] -= rng.uniform(0.006, 0.018)
            slope[t] -= rng.uniform(0.002, 0.010)
            curvature[t] += rng.normal(0.000, 0.006)
        elif u < 0.016:
            regime[t] = "bear_steepener"
            level[t] += rng.uniform(0.002, 0.010)
            slope[t] += rng.uniform(0.008, 0.018)
        elif u < 0.021:
            regime[t] = "bull_flattener"
            level[t] -= rng.uniform(0.002, 0.010)
            slope[t] -= rng.uniform(0.008, 0.018)

    slope_loading = (tenors - tenors.mean()) / tenors.max()
    curvature_loading = -((tenors - 7.0) / 10.0) ** 2 + 0.6
    curvature_loading = curvature_loading / np.max(np.abs(curvature_loading))

    curve_rows = []

    for i, date in enumerate(dates):
        curve = base_curve.values + level[i] + slope[i] * slope_loading + curvature[i] * curvature_loading
        curve = np.maximum(curve, 0.001)

        row = {"date": date, "level": level[i], "slope": slope[i], "curvature": curvature[i], "regime": regime[i]}
        for tenor, rate in zip(tenors, curve):
            row[f"{tenor:g}Y"] = rate
        curve_rows.append(row)

    curves = pd.DataFrame(curve_rows)
    curve_cols = [f"{t:g}Y" for t in tenors]

    tenor_metadata = pd.DataFrame({
        "tenor_years": tenors,
        "column": curve_cols,
        "base_zero_rate": base_curve.values,
        "slope_loading": slope_loading,
        "curvature_loading": curvature_loading,
    })

    return curves, tenor_metadata


curves_df, tenor_metadata = simulate_yield_curve_history(config)
curve_cols = tenor_metadata["column"].tolist()
tenors = tenor_metadata["tenor_years"].to_numpy()

curves_df.head(), tenor_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for col in ["0.25Y", "2Y", "5Y", "10Y", "30Y"]:
    plt.plot(curves_df["date"], curves_df[col], label=col)
plt.title("Synthetic Zero Curve History")
plt.xlabel("Date")
plt.ylabel("Zero rate")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
latest_curve = curves_df.iloc[-1][curve_cols].astype(float).to_numpy()
plt.plot(tenors, latest_curve, marker="o")
plt.title("Latest Synthetic Zero Curve")
plt.xlabel("Tenor years")
plt.ylabel("Zero rate")
plt.show()

## 8. Define a bond portfolio

We create a portfolio with:

- short Treasury note;
- belly bonds;
- long bonds;
- corporate-style spread bonds;
- inflation-sensitive proxy.

Each bond has:

1. maturity;
2. coupon;
3. frequency;
4. face amount;
5. quantity;
6. spread;
7. sector.

In [ ]:
bond_portfolio = pd.DataFrame([
    {"bond_id": "UST_2Y_LOWCOUPON", "sector": "sovereign", "maturity_years": 2.0,  "coupon_rate": 0.018, "frequency": 2, "face": 100.0, "quantity": 18_000, "z_spread_bp": 0.0},
    {"bond_id": "UST_5Y_CURRENT",   "sector": "sovereign", "maturity_years": 5.0,  "coupon_rate": 0.030, "frequency": 2, "face": 100.0, "quantity": 22_000, "z_spread_bp": 0.0},
    {"bond_id": "UST_10Y_CURRENT",  "sector": "sovereign", "maturity_years": 10.0, "coupon_rate": 0.035, "frequency": 2, "face": 100.0, "quantity": 16_000, "z_spread_bp": 0.0},
    {"bond_id": "UST_30Y_LONG",     "sector": "sovereign", "maturity_years": 30.0, "coupon_rate": 0.040, "frequency": 2, "face": 100.0, "quantity": 7_000,  "z_spread_bp": 0.0},
    {"bond_id": "IG_7Y_CORP",       "sector": "credit",    "maturity_years": 7.0,  "coupon_rate": 0.047, "frequency": 2, "face": 100.0, "quantity": 12_000, "z_spread_bp": 95.0},
    {"bond_id": "IG_12Y_CORP",      "sector": "credit",    "maturity_years": 12.0, "coupon_rate": 0.052, "frequency": 2, "face": 100.0, "quantity": 8_000,  "z_spread_bp": 125.0},
    {"bond_id": "HY_5Y_PROXY",      "sector": "high_yield","maturity_years": 5.0,  "coupon_rate": 0.075, "frequency": 2, "face": 100.0, "quantity": 6_000,  "z_spread_bp": 420.0},
    {"bond_id": "TIPS_10Y_PROXY",   "sector": "inflation", "maturity_years": 10.0, "coupon_rate": 0.020, "frequency": 2, "face": 100.0, "quantity": 8_000,  "z_spread_bp": -40.0},
])

bond_portfolio

## 9. Pricing functions

We build reusable functions for:

1. cashflow schedules;
2. zero-rate interpolation;
3. discount factors;
4. bond price;
5. yield to maturity.

In [ ]:
def cashflow_schedule(maturity_years, coupon_rate, frequency, face=100.0):
    n = int(round(maturity_years * frequency))
    times = np.arange(1, n + 1) / frequency
    coupon = coupon_rate * face / frequency
    cashflows = np.full(n, coupon, dtype=float)
    cashflows[-1] += face
    return times, cashflows


def curve_series_from_row(curve_row):
    values = curve_row[curve_cols].astype(float).to_numpy()
    return pd.Series(values, index=tenors)


def interpolate_zero_rates(curve_rates, times):
    curve_rates = pd.Series(curve_rates, index=tenors) if not isinstance(curve_rates, pd.Series) else curve_rates
    return np.interp(times, curve_rates.index.astype(float), curve_rates.values.astype(float))


def bond_price_from_curve(maturity_years, coupon_rate, frequency, curve_rates, face=100.0, z_spread_bp=0.0):
    times, cashflows = cashflow_schedule(maturity_years, coupon_rate, frequency, face)
    zero_rates = interpolate_zero_rates(curve_rates, times)
    spread = z_spread_bp / 10000.0
    discount_factors = np.exp(-(zero_rates + spread) * times)
    return float(np.sum(cashflows * discount_factors))


def yield_to_maturity(price, maturity_years, coupon_rate, frequency, face=100.0, low=-0.05, high=0.30, max_iter=200):
    times, cashflows = cashflow_schedule(maturity_years, coupon_rate, frequency, face)

    def price_from_y(y):
        return np.sum(cashflows / (1 + y / frequency) ** (frequency * times))

    lo, hi = low, high
    p_lo = price_from_y(lo) - price
    p_hi = price_from_y(hi) - price

    # Expand bounds if needed.
    for _ in range(20):
        if p_lo * p_hi <= 0:
            break
        hi += 0.10
        p_hi = price_from_y(hi) - price

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        p_mid = price_from_y(mid) - price

        if abs(p_mid) < 1e-10:
            return float(mid)

        if p_lo * p_mid <= 0:
            hi = mid
            p_hi = p_mid
        else:
            lo = mid
            p_lo = p_mid

    return float(0.5 * (lo + hi))

## 10. Price the portfolio on the latest curve

In [ ]:
latest_curve = curve_series_from_row(curves_df.iloc[-1])

pricing_rows = []

for _, bond in bond_portfolio.iterrows():
    price = bond_price_from_curve(
        maturity_years=bond["maturity_years"],
        coupon_rate=bond["coupon_rate"],
        frequency=int(bond["frequency"]),
        curve_rates=latest_curve,
        face=bond["face"],
        z_spread_bp=bond["z_spread_bp"]
    )

    ytm = yield_to_maturity(
        price=price,
        maturity_years=bond["maturity_years"],
        coupon_rate=bond["coupon_rate"],
        frequency=int(bond["frequency"]),
        face=bond["face"]
    )

    market_value = price * bond["quantity"]

    pricing_rows.append({
        "bond_id": bond["bond_id"],
        "sector": bond["sector"],
        "price": price,
        "ytm": ytm,
        "quantity": bond["quantity"],
        "market_value": market_value,
        "portfolio_weight": np.nan,
    })

bond_pricing = pd.DataFrame(pricing_rows)
bond_pricing["portfolio_weight"] = bond_pricing["market_value"] / bond_pricing["market_value"].sum()

bond_pricing

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = bond_pricing.sort_values("portfolio_weight")
plt.barh(plot_df["bond_id"], plot_df["portfolio_weight"])
plt.title("Portfolio Market Value Weights")
plt.xlabel("Weight")
plt.ylabel("Bond")
plt.show()

## 11. Duration and convexity functions

We compute:

1. Macaulay duration;
2. modified duration;
3. analytic yield convexity;
4. effective duration from curve bump;
5. effective convexity from curve bump;
6. DV01.

In [ ]:
def macaulay_duration(price, maturity_years, coupon_rate, frequency, ytm, face=100.0):
    times, cashflows = cashflow_schedule(maturity_years, coupon_rate, frequency, face)
    dfs = 1.0 / (1.0 + ytm / frequency) ** (frequency * times)
    pv = cashflows * dfs
    return float(np.sum(times * pv) / price)


def modified_duration(macaulay, ytm, frequency):
    return float(macaulay / (1.0 + ytm / frequency))


def yield_convexity(price, maturity_years, coupon_rate, frequency, ytm, face=100.0):
    times, cashflows = cashflow_schedule(maturity_years, coupon_rate, frequency, face)
    n = frequency * times
    dfs = 1.0 / (1.0 + ytm / frequency) ** n
    # Approx annual convexity for periodic yield.
    convexity = np.sum(cashflows * dfs * n * (n + 1)) / (price * frequency**2 * (1 + ytm / frequency)**2)
    return float(convexity)


def bumped_curve(curve_rates, bump_bp=1.0, key_tenor=None):
    bumped = curve_rates.copy()
    bump = bump_bp / 10000.0

    if key_tenor is None:
        bumped = bumped + bump
    else:
        # Triangular local bump around key tenor.
        distances = np.abs(bumped.index.astype(float) - key_tenor)
        scale = np.maximum(0.0, 1.0 - distances / max(key_tenor * 0.75, 1.0))
        if scale.max() > 0:
            scale = scale / scale.max()
        bumped = bumped + bump * scale

    return bumped


def effective_duration_convexity(bond, curve_rates, bump_bp=1.0):
    p0 = bond_price_from_curve(
        bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]),
        curve_rates, bond["face"], bond["z_spread_bp"]
    )

    p_down = bond_price_from_curve(
        bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]),
        bumped_curve(curve_rates, -bump_bp), bond["face"], bond["z_spread_bp"]
    )

    p_up = bond_price_from_curve(
        bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]),
        bumped_curve(curve_rates, bump_bp), bond["face"], bond["z_spread_bp"]
    )

    dy = bump_bp / 10000.0

    eff_duration = (p_down - p_up) / (2 * p0 * dy)
    eff_convexity = (p_down + p_up - 2 * p0) / (p0 * dy**2)

    dv01 = (p_down - p_up) / 2.0

    return float(eff_duration), float(eff_convexity), float(dv01)

In [ ]:
risk_rows = []

for _, bond in bond_portfolio.iterrows():
    price = bond_pricing.loc[bond_pricing["bond_id"] == bond["bond_id"], "price"].iloc[0]
    ytm = bond_pricing.loc[bond_pricing["bond_id"] == bond["bond_id"], "ytm"].iloc[0]

    mac = macaulay_duration(price, bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]), ytm, bond["face"])
    mod = modified_duration(mac, ytm, int(bond["frequency"]))
    conv = yield_convexity(price, bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]), ytm, bond["face"])
    eff_dur, eff_conv, dv01_price = effective_duration_convexity(bond, latest_curve, config.bump_bp)

    market_value = price * bond["quantity"]
    dv01_dollar = dv01_price * bond["quantity"]

    risk_rows.append({
        "bond_id": bond["bond_id"],
        "sector": bond["sector"],
        "price": price,
        "ytm": ytm,
        "macaulay_duration": mac,
        "modified_duration": mod,
        "yield_convexity": conv,
        "effective_duration": eff_dur,
        "effective_convexity": eff_conv,
        "price_DV01": dv01_price,
        "dollar_DV01": dv01_dollar,
        "market_value": market_value,
    })

bond_risk = pd.DataFrame(risk_rows)
bond_risk["portfolio_weight"] = bond_risk["market_value"] / bond_risk["market_value"].sum()
bond_risk["portfolio_DV01_contribution_pct"] = bond_risk["dollar_DV01"] / bond_risk["dollar_DV01"].sum()

bond_risk.sort_values("dollar_DV01", ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = bond_risk.sort_values("effective_duration")
plt.barh(plot_df["bond_id"], plot_df["effective_duration"])
plt.title("Effective Duration by Bond")
plt.xlabel("Effective duration")
plt.ylabel("Bond")
plt.show()

plt.figure(figsize=(10, 6))
plot_df = bond_risk.sort_values("dollar_DV01")
plt.barh(plot_df["bond_id"], plot_df["dollar_DV01"])
plt.title("Dollar DV01 by Bond")
plt.xlabel("Dollar DV01 per 1 bp")
plt.ylabel("Bond")
plt.show()

## 12. Portfolio duration, convexity, and DV01

Portfolio effective duration is market-value weighted:

$$
D_p = \sum_i w_iD_i
$$

Portfolio convexity:

$$
C_p = \sum_i w_iC_i
$$

Portfolio DV01:

$$
DV01_p = \sum_i DV01_i
$$

In [ ]:
portfolio_market_value = bond_risk["market_value"].sum()
portfolio_effective_duration = float(np.sum(bond_risk["portfolio_weight"] * bond_risk["effective_duration"]))
portfolio_modified_duration = float(np.sum(bond_risk["portfolio_weight"] * bond_risk["modified_duration"]))
portfolio_convexity = float(np.sum(bond_risk["portfolio_weight"] * bond_risk["effective_convexity"]))
portfolio_dv01 = float(bond_risk["dollar_DV01"].sum())

portfolio_risk_summary = pd.Series({
    "portfolio_market_value": portfolio_market_value,
    "portfolio_modified_duration": portfolio_modified_duration,
    "portfolio_effective_duration": portfolio_effective_duration,
    "portfolio_effective_convexity": portfolio_convexity,
    "portfolio_DV01": portfolio_dv01,
    "portfolio_DV01_per_1mm_market_value": portfolio_dv01 / portfolio_market_value * 1_000_000,
})

portfolio_risk_summary

## 13. Key-rate duration

Parallel duration misses curve-shape risk.

We compute key-rate DV01 and key-rate duration by bumping each tenor locally.

In [ ]:
def bond_key_rate_dv01(bond, curve_rates, key_tenors, bump_bp=1.0):
    p0 = bond_price_from_curve(
        bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]),
        curve_rates, bond["face"], bond["z_spread_bp"]
    )

    rows = []

    for key in key_tenors:
        p_down = bond_price_from_curve(
            bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]),
            bumped_curve(curve_rates, -bump_bp, key_tenor=key), bond["face"], bond["z_spread_bp"]
        )
        p_up = bond_price_from_curve(
            bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]),
            bumped_curve(curve_rates, bump_bp, key_tenor=key), bond["face"], bond["z_spread_bp"]
        )

        dy = bump_bp / 10000.0
        dv01 = (p_down - p_up) / 2.0
        krd = (p_down - p_up) / (2 * p0 * dy)

        rows.append({
            "bond_id": bond["bond_id"],
            "key_tenor": key,
            "key_rate_DV01_price": dv01,
            "key_rate_duration": krd,
            "quantity": bond["quantity"],
            "key_rate_DV01_dollar": dv01 * bond["quantity"],
        })

    return pd.DataFrame(rows)


key_tenors = np.array([1, 2, 3, 5, 7, 10, 20, 30], dtype=float)

krd_frames = []
for _, bond in bond_portfolio.iterrows():
    krd_frames.append(bond_key_rate_dv01(bond, latest_curve, key_tenors, config.bump_bp))

key_rate_table = pd.concat(krd_frames, ignore_index=True)

portfolio_key_rate = (
    key_rate_table
    .groupby("key_tenor")
    .agg(
        portfolio_key_rate_DV01=("key_rate_DV01_dollar", "sum")
    )
    .reset_index()
)

portfolio_key_rate["portfolio_key_rate_duration"] = portfolio_key_rate["portfolio_key_rate_DV01"] / portfolio_market_value / (config.bump_bp / 10000.0)

portfolio_key_rate

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(portfolio_key_rate["key_tenor"].astype(str), portfolio_key_rate["portfolio_key_rate_DV01"])
plt.title("Portfolio Key-Rate DV01")
plt.xlabel("Key tenor")
plt.ylabel("Dollar DV01")
plt.show()

## 14. Curve shock scenarios

We create curve shocks:

1. parallel up;
2. parallel down;
3. bear steepener;
4. bull flattener;
5. belly cheapening;
6. recession rally;
7. inflation shock.

For each scenario, we fully reprice the bond portfolio.

In [ ]:
def build_curve_shocks(key_tenors):
    shocks = {}

    shocks["parallel_up_100bp"] = pd.Series(0.0100, index=tenors)
    shocks["parallel_down_100bp"] = pd.Series(-0.0100, index=tenors)

    slope_loading = (tenors - tenors.mean()) / tenors.max()
    shocks["bear_steepener"] = pd.Series(0.004 + 0.012 * slope_loading, index=tenors)
    shocks["bull_flattener"] = pd.Series(-0.004 - 0.012 * slope_loading, index=tenors)

    belly = np.exp(-0.5 * ((tenors - 5.0) / 2.0) ** 2)
    shocks["belly_cheapening"] = pd.Series(0.010 * belly, index=tenors)

    shocks["recession_rally"] = pd.Series(-0.010 + 0.004 * slope_loading, index=tenors)
    shocks["inflation_shock"] = pd.Series(0.008 + 0.006 * np.maximum(slope_loading, 0), index=tenors)

    return pd.DataFrame(shocks).T


curve_shocks = build_curve_shocks(key_tenors)
curve_shocks.columns = [f"{t:g}Y" for t in tenors]

curve_shocks

In [ ]:
def price_portfolio_given_curve(bond_portfolio, curve_rates):
    rows = []
    total_value = 0.0

    for _, bond in bond_portfolio.iterrows():
        price = bond_price_from_curve(
            bond["maturity_years"], bond["coupon_rate"], int(bond["frequency"]),
            curve_rates, bond["face"], bond["z_spread_bp"]
        )
        value = price * bond["quantity"]
        total_value += value

        rows.append({
            "bond_id": bond["bond_id"],
            "price": price,
            "market_value": value,
        })

    return total_value, pd.DataFrame(rows)


base_value, base_price_detail = price_portfolio_given_curve(bond_portfolio, latest_curve)

scenario_rows = []
scenario_detail_frames = []

for scenario, shock_row in curve_shocks.iterrows():
    shock = pd.Series(shock_row.values.astype(float), index=tenors)
    shocked_curve = latest_curve + shock

    value, detail = price_portfolio_given_curve(bond_portfolio, shocked_curve)
    pnl = value - base_value

    scenario_rows.append({
        "scenario": scenario,
        "portfolio_value": value,
        "portfolio_pnl": pnl,
        "portfolio_return": pnl / base_value,
    })

    detail["scenario"] = scenario
    detail = detail.merge(base_price_detail.rename(columns={"price": "base_price", "market_value": "base_market_value"}), on="bond_id", how="left")
    detail["pnl"] = detail["market_value"] - detail["base_market_value"]
    scenario_detail_frames.append(detail)

scenario_results = pd.DataFrame(scenario_rows).sort_values("portfolio_pnl")
scenario_details = pd.concat(scenario_detail_frames, ignore_index=True)

scenario_results

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = scenario_results.sort_values("portfolio_pnl")
plt.barh(plot_df["scenario"], plot_df["portfolio_pnl"])
plt.axvline(0, linestyle="--")
plt.title("Curve Shock Scenario PnL")
plt.xlabel("Portfolio PnL")
plt.ylabel("Scenario")
plt.show()

## 15. Duration-convexity approximation

For a parallel yield shock $\Delta y$:

$$
\Delta P \approx P \Big[ -D\Delta y + \frac{1}{2}C(\Delta y)^2 \Big]
$$

We compare:

1. full revaluation;
2. duration-only approximation;
3. duration-plus-convexity approximation.

In [ ]:
def duration_convexity_approx(value, duration, convexity, dy):
    return value * (-duration * dy + 0.5 * convexity * dy**2)


approx_rows = []

for shock_bp in [-200, -100, -50, 50, 100, 200]:
    dy = shock_bp / 10000.0
    shocked_curve = latest_curve + dy
    full_value, _ = price_portfolio_given_curve(bond_portfolio, shocked_curve)
    full_pnl = full_value - base_value

    dur_only = base_value * (-portfolio_effective_duration * dy)
    dur_conv = duration_convexity_approx(base_value, portfolio_effective_duration, portfolio_convexity, dy)

    approx_rows.append({
        "shock_bp": shock_bp,
        "full_revaluation_pnl": full_pnl,
        "duration_only_pnl": dur_only,
        "duration_convexity_pnl": dur_conv,
        "duration_only_error": dur_only - full_pnl,
        "duration_convexity_error": dur_conv - full_pnl,
    })

duration_convexity_approx_report = pd.DataFrame(approx_rows)

duration_convexity_approx_report

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(duration_convexity_approx_report["shock_bp"], duration_convexity_approx_report["full_revaluation_pnl"], marker="o", label="full revaluation")
plt.plot(duration_convexity_approx_report["shock_bp"], duration_convexity_approx_report["duration_only_pnl"], marker="x", label="duration only")
plt.plot(duration_convexity_approx_report["shock_bp"], duration_convexity_approx_report["duration_convexity_pnl"], marker="s", label="duration + convexity")
plt.axhline(0, linestyle="--")
plt.title("Duration-Convexity Approximation")
plt.xlabel("Parallel shock bp")
plt.ylabel("Portfolio PnL")
plt.legend()
plt.show()

## 16. Barbell versus bullet comparison

A bullet portfolio concentrates around one maturity.

A barbell portfolio combines short and long bonds.

They can have similar duration but different convexity and key-rate risk.

In [ ]:
def make_strategy_weights(strategy_name):
    weights = pd.Series(0.0, index=bond_portfolio["bond_id"])

    if strategy_name == "bullet_10y":
        weights["UST_10Y_CURRENT"] = 0.55
        weights["IG_12Y_CORP"] = 0.25
        weights["TIPS_10Y_PROXY"] = 0.20
    elif strategy_name == "barbell_2y_30y":
        weights["UST_2Y_LOWCOUPON"] = 0.55
        weights["UST_30Y_LONG"] = 0.30
        weights["GOLD_PLACEHOLDER"] = 0.0 if "GOLD_PLACEHOLDER" in weights.index else 0.0
        weights["UST_5Y_CURRENT"] = 0.15
    elif strategy_name == "credit_belly":
        weights["IG_7Y_CORP"] = 0.45
        weights["HY_5Y_PROXY"] = 0.30
        weights["UST_5Y_CURRENT"] = 0.25
    else:
        raise ValueError("Unknown strategy.")

    weights = weights / weights.sum()
    return weights


strategy_names = ["bullet_10y", "barbell_2y_30y", "credit_belly"]

strategy_quantity_frames = []
strategy_reports = []

for strategy in strategy_names:
    value_weights = make_strategy_weights(strategy)

    strategy_port = bond_portfolio.copy()
    strategy_port["target_value_weight"] = strategy_port["bond_id"].map(value_weights).fillna(0.0)

    # Convert target value weights into quantities based on current price and same base value.
    strategy_port = strategy_port.merge(bond_pricing[["bond_id", "price"]], on="bond_id", how="left")
    strategy_port["target_market_value"] = base_value * strategy_port["target_value_weight"]
    strategy_port["quantity"] = np.where(
        strategy_port["price"] > 0,
        strategy_port["target_market_value"] / strategy_port["price"],
        0.0
    )

    strategy_value, _ = price_portfolio_given_curve(strategy_port, latest_curve)

    strategy_risk_rows = []
    for _, bond in strategy_port.iterrows():
        if bond["quantity"] == 0:
            continue
        eff_dur, eff_conv, dv01_price = effective_duration_convexity(bond, latest_curve, config.bump_bp)
        price = bond["price"]
        mv = price * bond["quantity"]
        strategy_risk_rows.append({
            "bond_id": bond["bond_id"],
            "market_value": mv,
            "effective_duration": eff_dur,
            "effective_convexity": eff_conv,
            "dollar_DV01": dv01_price * bond["quantity"],
        })

    sr = pd.DataFrame(strategy_risk_rows)
    duration = np.sum(sr["market_value"] / sr["market_value"].sum() * sr["effective_duration"])
    convexity = np.sum(sr["market_value"] / sr["market_value"].sum() * sr["effective_convexity"])
    dv01 = sr["dollar_DV01"].sum()

    strategy_reports.append({
        "strategy": strategy,
        "market_value": strategy_value,
        "effective_duration": duration,
        "effective_convexity": convexity,
        "dollar_DV01": dv01,
    })

    strategy_quantity_frames.append(strategy_port.assign(strategy=strategy))

strategy_report = pd.DataFrame(strategy_reports)

strategy_report

## 17. Liability immunisation

Suppose we must fund a liability at a future date.

A simple duration immunisation objective:

1. match present value;
2. match duration;
3. maximise or prefer convexity;
4. obey long-only constraints.

We solve a small optimisation problem over available bonds.

In [ ]:
liability_maturity = 8.0
liability_cashflow = 5_000_000.0
liability_discount_rate = interpolate_zero_rates(latest_curve, np.array([liability_maturity]))[0]
liability_pv = liability_cashflow * np.exp(-liability_discount_rate * liability_maturity)
liability_duration = liability_maturity

liability_report = pd.Series({
    "liability_cashflow": liability_cashflow,
    "liability_maturity": liability_maturity,
    "liability_zero_rate": liability_discount_rate,
    "liability_present_value": liability_pv,
    "liability_duration": liability_duration,
})

liability_report

In [ ]:
def immunisation_objective(weights, bond_risk, target_duration, penalty_weight=100.0):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    duration = weights @ bond_risk["effective_duration"].to_numpy()
    convexity = weights @ bond_risk["effective_convexity"].to_numpy()

    # Match duration while preferring convexity.
    return penalty_weight * (duration - target_duration) ** 2 - 0.0001 * convexity


def solve_immunisation(bond_risk, target_duration):
    n = len(bond_risk)
    x0 = np.ones(n) / n

    if SCIPY_AVAILABLE:
        res = minimize(
            lambda w: immunisation_objective(w, bond_risk, target_duration),
            x0=x0,
            method="SLSQP",
            bounds=[(0.0, 0.50)] * n,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}],
            options={"maxiter": 1000, "ftol": 1e-12}
        )
        if res.success:
            return pd.Series(res.x / res.x.sum(), index=bond_risk["bond_id"]), {
                "success": True,
                "method": "scipy_slsqp",
                "message": res.message,
            }

    # fallback grid-ish random search
    rng = np.random.default_rng(config.seed + 99)
    best = x0
    best_val = immunisation_objective(best, bond_risk, target_duration)
    for _ in range(50_000):
        w = rng.dirichlet(np.ones(n))
        w = np.clip(w, 0.0, 0.50)
        w = w / w.sum()
        val = immunisation_objective(w, bond_risk, target_duration)
        if val < best_val:
            best = w
            best_val = val

    return pd.Series(best, index=bond_risk["bond_id"]), {
        "success": False,
        "method": "fallback_random_search",
        "message": "scipy unavailable or failed",
    }


immunisation_weights, immunisation_info = solve_immunisation(bond_risk, liability_duration)

immunisation_table = pd.DataFrame({
    "bond_id": immunisation_weights.index,
    "immunisation_weight": immunisation_weights.values,
}).merge(bond_risk[["bond_id", "effective_duration", "effective_convexity"]], on="bond_id", how="left")

immunisation_duration = float((immunisation_table["immunisation_weight"] * immunisation_table["effective_duration"]).sum())
immunisation_convexity = float((immunisation_table["immunisation_weight"] * immunisation_table["effective_convexity"]).sum())

immunisation_summary = pd.Series({
    "target_liability_duration": liability_duration,
    "immunised_asset_duration": immunisation_duration,
    "duration_gap": immunisation_duration - liability_duration,
    "immunised_asset_convexity": immunisation_convexity,
    "solver_method": immunisation_info["method"],
    "solver_success": immunisation_info["success"],
})

immunisation_table, immunisation_summary

## 18. Futures-style duration hedge

If a futures contract has DV01:

$$
DV01_F
$$

and the portfolio has DV01:

$$
DV01_P
$$

To reduce portfolio DV01 to target:

$$
N = -\frac{DV01_P-DV01_{target}}{DV01_F}
$$

where $N$ is signed contract count.

A negative number means short futures if the portfolio is long duration.

In [ ]:
# Create a simple deliverable futures proxy: 10Y Treasury futures approximated by UST_10Y_CURRENT.
futures_proxy_bond = bond_portfolio[bond_portfolio["bond_id"] == "UST_10Y_CURRENT"].iloc[0].copy()
futures_proxy_bond["quantity"] = 1.0

fut_eff_dur, fut_eff_conv, fut_dv01_price = effective_duration_convexity(futures_proxy_bond, latest_curve, config.bump_bp)

# Futures contract DV01 scales by price, multiplier and quote convention.
# Here we approximate contract DV01 as DV01 per 100 face times multiplier.
futures_contract_dv01 = fut_dv01_price * config.futures_multiplier

target_portfolio_dv01 = 0.0
futures_contracts_needed = -(portfolio_dv01 - target_portfolio_dv01) / futures_contract_dv01

futures_hedge_report = pd.Series({
    "futures_proxy": "UST_10Y_CURRENT",
    "futures_price": config.futures_price,
    "futures_multiplier": config.futures_multiplier,
    "futures_proxy_effective_duration": fut_eff_dur,
    "futures_contract_DV01": futures_contract_dv01,
    "portfolio_DV01": portfolio_dv01,
    "target_portfolio_DV01": target_portfolio_dv01,
    "signed_futures_contracts_needed": futures_contracts_needed,
    "rounded_contracts": int(np.round(futures_contracts_needed)),
})

futures_hedge_report

## 19. Hedge performance under curve scenarios

A duration hedge with a 10-year futures proxy mainly hedges the 10-year area.

It can leave residual risk under steepeners, flatteners, and belly shocks.

In [ ]:
rounded_contracts = int(np.round(futures_contracts_needed))

hedge_scenario_rows = []

for scenario, shock_row in curve_shocks.iterrows():
    shock = pd.Series(shock_row.values.astype(float), index=tenors)
    shocked_curve = latest_curve + shock

    unhedged_value, _ = price_portfolio_given_curve(bond_portfolio, shocked_curve)
    unhedged_pnl = unhedged_value - base_value

    fut_base_price = bond_price_from_curve(
        futures_proxy_bond["maturity_years"], futures_proxy_bond["coupon_rate"], int(futures_proxy_bond["frequency"]),
        latest_curve, futures_proxy_bond["face"], futures_proxy_bond["z_spread_bp"]
    )
    fut_shocked_price = bond_price_from_curve(
        futures_proxy_bond["maturity_years"], futures_proxy_bond["coupon_rate"], int(futures_proxy_bond["frequency"]),
        shocked_curve, futures_proxy_bond["face"], futures_proxy_bond["z_spread_bp"]
    )

    futures_pnl = rounded_contracts * (fut_shocked_price - fut_base_price) * config.futures_multiplier
    hedged_pnl = unhedged_pnl + futures_pnl

    hedge_scenario_rows.append({
        "scenario": scenario,
        "unhedged_pnl": unhedged_pnl,
        "futures_pnl": futures_pnl,
        "hedged_pnl": hedged_pnl,
        "hedge_effectiveness_pnl_reduction": 1 - abs(hedged_pnl) / abs(unhedged_pnl) if abs(unhedged_pnl) > 0 else np.nan,
    })

hedge_scenario_report = pd.DataFrame(hedge_scenario_rows).sort_values("hedged_pnl")

hedge_scenario_report

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(len(hedge_scenario_report))
plt.bar(x - 0.2, hedge_scenario_report["unhedged_pnl"], width=0.4, label="unhedged")
plt.bar(x + 0.2, hedge_scenario_report["hedged_pnl"], width=0.4, label="hedged")
plt.axhline(0, linestyle="--")
plt.xticks(x, hedge_scenario_report["scenario"], rotation=45, ha="right")
plt.title("Unhedged vs Futures-Hedged Scenario PnL")
plt.ylabel("PnL")
plt.legend()
plt.tight_layout()
plt.show()

## 20. Rolling portfolio valuation through historical curves

We value the same bond portfolio across the simulated curve history.

This gives a curve-driven PnL series.

In [ ]:
valuation_rows = []

for _, curve_row in curves_df.iterrows():
    curve = curve_series_from_row(curve_row)
    value, _ = price_portfolio_given_curve(bond_portfolio, curve)
    valuation_rows.append({
        "date": curve_row["date"],
        "portfolio_value": value,
        "regime": curve_row["regime"],
        "level": curve_row["level"],
        "slope": curve_row["slope"],
        "curvature": curve_row["curvature"],
    })

portfolio_valuation_history = pd.DataFrame(valuation_rows)
portfolio_valuation_history["daily_pnl"] = portfolio_valuation_history["portfolio_value"].diff()
portfolio_valuation_history["daily_return"] = portfolio_valuation_history["portfolio_value"].pct_change()

portfolio_valuation_history.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(portfolio_valuation_history["date"], portfolio_valuation_history["portfolio_value"])
plt.title("Portfolio Value Across Simulated Curve History")
plt.xlabel("Date")
plt.ylabel("Portfolio value")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(portfolio_valuation_history["daily_return"].dropna(), bins=100)
plt.title("Curve-Driven Daily Return Distribution")
plt.xlabel("Daily return")
plt.ylabel("Count")
plt.show()

## 21. Factor attribution: level, slope, curvature

We regress daily portfolio returns on curve factor changes.

This estimates sensitivity to:

1. level;
2. slope;
3. curvature.

In [ ]:
factor_df = portfolio_valuation_history[["date", "daily_return", "level", "slope", "curvature"]].dropna().copy()
factor_df[["d_level", "d_slope", "d_curvature"]] = factor_df[["level", "slope", "curvature"]].diff()
factor_df = factor_df.dropna()

X = factor_df[["d_level", "d_slope", "d_curvature"]].to_numpy()
y = factor_df["daily_return"].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])
beta = np.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y
fitted = X_design @ beta
resid = y - fitted
r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))

factor_sensitivity = pd.Series({
    "intercept": beta[0],
    "level_beta": beta[1],
    "slope_beta": beta[2],
    "curvature_beta": beta[3],
    "r_squared": r2,
})

factor_sensitivity

## 22. Tail risk from curve-driven PnL

We compute VaR and CVaR of curve-driven daily returns.

In [ ]:
def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


daily_returns = portfolio_valuation_history["daily_return"].dropna()
losses = -daily_returns

tail_rows = []
for alpha in [0.95, 0.99]:
    var, cvar = historical_var_cvar(losses, alpha)
    tail_rows.append({
        "alpha": alpha,
        "VaR_return": var,
        "CVaR_return": cvar,
        "VaR_dollar": var * portfolio_market_value,
        "CVaR_dollar": cvar * portfolio_market_value,
    })

tail_risk_report = pd.DataFrame(tail_rows)

tail_risk_report

## 23. Governance flags

Fixed-income risk should be reviewed if:

1. portfolio DV01 is too large;
2. one key-rate bucket dominates;
3. scenario loss exceeds threshold;
4. hedge leaves large residual PnL;
5. convexity approximation error is large;
6. liability duration gap is material.

In [ ]:
max_key_rate_share = float(portfolio_key_rate["portfolio_key_rate_DV01"].abs().max() / portfolio_key_rate["portfolio_key_rate_DV01"].abs().sum())
worst_scenario_loss = float(scenario_results["portfolio_pnl"].min())
worst_hedged_scenario_loss = float(hedge_scenario_report["hedged_pnl"].min())
max_duration_convexity_error = float(duration_convexity_approx_report["duration_convexity_error"].abs().max())

governance_flags = pd.DataFrame([{
    "portfolio_DV01": portfolio_dv01,
    "portfolio_effective_duration": portfolio_effective_duration,
    "max_key_rate_DV01_share": max_key_rate_share,
    "worst_scenario_loss": worst_scenario_loss,
    "worst_hedged_scenario_loss": worst_hedged_scenario_loss,
    "max_duration_convexity_approx_error": max_duration_convexity_error,
    "liability_duration_gap": immunisation_duration - liability_duration,
    "flag_DV01_above_5000": bool(abs(portfolio_dv01) > 5000),
    "flag_key_rate_concentration_above_40pct": bool(max_key_rate_share > 0.40),
    "flag_worst_scenario_loss_above_5pct_mv": bool(abs(worst_scenario_loss) > 0.05 * portfolio_market_value),
    "flag_hedged_loss_above_3pct_mv": bool(abs(worst_hedged_scenario_loss) > 0.03 * portfolio_market_value),
    "flag_duration_convexity_error_above_1pct_mv": bool(max_duration_convexity_error > 0.01 * portfolio_market_value),
    "flag_liability_duration_gap_above_0_25y": bool(abs(immunisation_duration - liability_duration) > 0.25),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_DV01_above_5000",
        "flag_key_rate_concentration_above_40pct",
        "flag_worst_scenario_loss_above_5pct_mv",
        "flag_hedged_loss_above_3pct_mv",
        "flag_duration_convexity_error_above_1pct_mv",
        "flag_liability_duration_gap_above_0_25y",
    ]
].any(axis=1)

governance_flags

## 24. Audit checks

Numerical checks:

1. all bond prices are positive;
2. portfolio market value is positive;
3. DV01 is positive for long fixed-rate bonds;
4. key-rate DV01 sums roughly to parallel DV01;
5. scenario pricing is finite;
6. liability PV is positive.

In [ ]:
audit_rows = []

audit_rows.append({
    "check": "bond_prices_positive",
    "value": float(bond_pricing["price"].min()),
    "passed": bool((bond_pricing["price"] > 0).all())
})

audit_rows.append({
    "check": "portfolio_market_value_positive",
    "value": float(portfolio_market_value),
    "passed": bool(portfolio_market_value > 0)
})

audit_rows.append({
    "check": "portfolio_DV01_positive",
    "value": float(portfolio_dv01),
    "passed": bool(portfolio_dv01 > 0)
})

krd_sum = portfolio_key_rate["portfolio_key_rate_DV01"].sum()
audit_rows.append({
    "check": "key_rate_DV01_sum_reasonable_vs_parallel_DV01",
    "value": float(krd_sum / portfolio_dv01 if portfolio_dv01 != 0 else np.nan),
    "passed": bool(0.50 < krd_sum / portfolio_dv01 < 1.80)
})

audit_rows.append({
    "check": "scenario_results_finite",
    "value": bool(np.isfinite(scenario_results["portfolio_pnl"]).all()),
    "passed": bool(np.isfinite(scenario_results["portfolio_pnl"]).all())
})

audit_rows.append({
    "check": "liability_present_value_positive",
    "value": float(liability_pv),
    "passed": bool(liability_pv > 0)
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 25. Practical checklist for fixed-income risk

Before trusting a bond risk report:

1. **Price from a curve**  
   Are cashflows and discount factors correct?

2. **Check yield conventions**  
   Compounding, day count, coupon frequency, and clean/dirty price matter.

3. **Report DV01**  
   Duration is abstract; DV01 is dollar risk.

4. **Use key-rate durations**  
   Parallel duration misses curve twists.

5. **Stress curve shapes**  
   Steepeners, flatteners, belly shocks, and recession rallies matter.

6. **Use convexity**  
   Duration-only approximations fail for large shocks.

7. **Separate rate and spread risk**  
   Credit bonds have Treasury risk plus spread risk.

8. **Check hedge basis**  
   Treasury futures hedge some but not all curve risk.

9. **Immunise liabilities carefully**  
   Match PV and duration, then monitor convexity and curve risk.

10. **Govern concentration**  
   One key-rate bucket can dominate the portfolio.

## 26. Saving outputs

The notebook saves:

1. simulated yield curves;
2. tenor metadata;
3. bond portfolio definition;
4. bond prices;
5. bond risk metrics;
6. portfolio risk summary;
7. key-rate duration table;
8. curve shocks;
9. scenario results;
10. scenario details;
11. duration-convexity approximation report;
12. barbell/bullet strategy report;
13. liability immunisation table;
14. futures hedge report;
15. hedge scenario report;
16. portfolio valuation history;
17. factor sensitivity;
18. tail risk report;
19. governance flags;
20. audit report;
21. manifest.

In [ ]:
output_dir = Path("data/processed/fixed_income_duration_convexity_risk_v1")
output_dir.mkdir(parents=True, exist_ok=True)

curves_path = output_dir / "simulated_yield_curves.csv"
tenor_metadata_path = output_dir / "tenor_metadata.csv"
bond_portfolio_path = output_dir / "bond_portfolio.csv"
bond_pricing_path = output_dir / "bond_pricing.csv"
bond_risk_path = output_dir / "bond_risk_metrics.csv"
portfolio_risk_path = output_dir / "portfolio_risk_summary.csv"
key_rate_path = output_dir / "key_rate_duration_table.csv"
portfolio_key_rate_path = output_dir / "portfolio_key_rate_dv01.csv"
curve_shocks_path = output_dir / "curve_shocks.csv"
scenario_results_path = output_dir / "scenario_results.csv"
scenario_details_path = output_dir / "scenario_details.csv"
duration_convexity_path = output_dir / "duration_convexity_approximation.csv"
strategy_report_path = output_dir / "barbell_bullet_strategy_report.csv"
liability_report_path = output_dir / "liability_report.csv"
immunisation_table_path = output_dir / "immunisation_table.csv"
immunisation_summary_path = output_dir / "immunisation_summary.csv"
futures_hedge_path = output_dir / "futures_hedge_report.csv"
hedge_scenario_path = output_dir / "hedge_scenario_report.csv"
valuation_history_path = output_dir / "portfolio_valuation_history.csv"
factor_sensitivity_path = output_dir / "factor_sensitivity.csv"
tail_risk_path = output_dir / "tail_risk_report.csv"
governance_flags_path = output_dir / "governance_flags.csv"
audit_report_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

curves_df.to_csv(curves_path, index=False)
tenor_metadata.to_csv(tenor_metadata_path, index=False)
bond_portfolio.to_csv(bond_portfolio_path, index=False)
bond_pricing.to_csv(bond_pricing_path, index=False)
bond_risk.to_csv(bond_risk_path, index=False)
portfolio_risk_summary.to_frame("value").to_csv(portfolio_risk_path)
key_rate_table.to_csv(key_rate_path, index=False)
portfolio_key_rate.to_csv(portfolio_key_rate_path, index=False)
curve_shocks.to_csv(curve_shocks_path)
scenario_results.to_csv(scenario_results_path, index=False)
scenario_details.to_csv(scenario_details_path, index=False)
duration_convexity_approx_report.to_csv(duration_convexity_path, index=False)
strategy_report.to_csv(strategy_report_path, index=False)
liability_report.to_frame("value").to_csv(liability_report_path)
immunisation_table.to_csv(immunisation_table_path, index=False)
immunisation_summary.to_frame("value").to_csv(immunisation_summary_path)
futures_hedge_report.to_frame("value").to_csv(futures_hedge_path)
hedge_scenario_report.to_csv(hedge_scenario_path, index=False)
portfolio_valuation_history.to_csv(valuation_history_path, index=False)
factor_sensitivity.to_frame("value").to_csv(factor_sensitivity_path)
tail_risk_report.to_csv(tail_risk_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)
audit_report.to_csv(audit_report_path, index=False)

manifest = {
    "dataset_name": "fixed_income_duration_convexity_risk_outputs",
    "schema_version": "fixed_income_duration_convexity_risk_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "curve_columns": curve_cols,
    "key_tenors": key_tenors.tolist(),
    "portfolio_risk_summary": portfolio_risk_summary.to_dict(),
    "futures_hedge_report": futures_hedge_report.to_dict(),
    "immunisation_summary": immunisation_summary.to_dict(),
    "tail_risk_report": tail_risk_report.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "Bond prices are computed from continuously compounded zero curves with optional z-spreads.",
        "Macaulay and modified duration are computed from yield to maturity.",
        "Effective duration and convexity are computed by bumping the zero curve.",
        "Key-rate duration uses local triangular bumps at selected curve tenors.",
        "Scenario analysis uses full revaluation, not only duration approximation.",
        "Futures duration hedge is approximated using a 10-year Treasury bond proxy.",
        "Liability immunisation matches duration using a constrained long-only optimisation.",
        "This notebook is educational; production fixed-income risk requires accrued interest, day count, clean/dirty pricing, calendars, embedded options, credit migration, liquidity and futures delivery-basket details."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

curves_path, bond_risk_path, scenario_results_path, governance_flags_path, manifest_path

## 27. Limitations

### 27.1 Synthetic yield curves

The yield curves are simulated and simplified.

Real curves require bootstrapping from deposits, futures, swaps, or Treasury securities.

### 27.2 Continuous discounting simplification

The notebook uses continuously compounded zero rates for pricing.

Market conventions use specific compounding, day-count, coupon, and settlement rules.

### 27.3 Clean versus dirty price

Accrued interest is ignored.

Production bond pricing must distinguish clean price, dirty price, and settlement date.

### 27.4 Spread risk simplified

Credit bonds use static z-spreads.

Real credit risk includes spread curve risk, default, migration, liquidity, and recovery.

### 27.5 Embedded options excluded

Callable, putable, mortgage, and prepayable bonds require option-adjusted spread and model-based duration.

### 27.6 Futures hedge approximation

The futures hedge uses a simplified 10-year bond proxy.

Real Treasury futures require conversion factors, cheapest-to-deliver, delivery options, and basis modelling.

### 27.7 Liability immunisation simplified

The liability is a single cashflow.

Real liability-driven investing uses full liability cashflow curves and surplus risk.

### 27.8 Key-rate bump design

Key-rate duration depends on bump interpolation method.

Different desks use different key-rate definitions.

## 28. Research frontier and extensions

Important extensions include:

1. **Curve bootstrapping engine**  
   Build zero curves from market instruments.

2. **Nelson-Siegel-Svensson calibration**  
   Fit smooth curves and factor risk.

3. **OAS and callable bond risk**  
   Add embedded optionality.

4. **Credit spread duration**  
   Separate Treasury DV01 and spread DV01.

5. **Inflation-linked bond analytics**  
   Real yield, breakeven, inflation carry.

6. **Treasury futures basis model**  
   CTD, conversion factors, implied repo, delivery option.

7. **Liability-driven investing**  
   Multi-period liability cashflows and surplus optimisation.

8. **Scenario engine**  
   Historical replay, macro scenarios, stress curves.

9. **PCA yield curve risk**  
   Level, slope, curvature factor shocks from historical curves.

10. **Chinese government bond futures / rates capstone**  
   Duration hedging, CTD basis, repo, and local curve dynamics.

## 29. Phase 4 closing note

Phase 4 built a portfolio construction and risk-management stack:

1. return accounting;
2. factor attribution;
3. volatility targeting;
4. covariance shrinkage;
5. PCA risk modes;
6. VaR/CVaR and stress testing;
7. beta and tail hedging;
8. futures hedge ratios;
9. risk parity;
10. HRP;
11. CVaR convex optimisation;
12. Black-Litterman;
13. stochastic control;
14. fixed-income duration and convexity.

This notebook closes the phase by adding fixed-income risk, where the main state variable is not just asset return but the entire yield curve.

## 30. Summary

This notebook implemented fixed-income duration, convexity, and curve-risk analytics.

It showed how to:

1. simulate zero-curve histories;
2. price coupon bonds from a curve;
3. compute yield to maturity;
4. compute Macaulay and modified duration;
5. compute effective duration and convexity;
6. calculate DV01/PV01;
7. build portfolio duration and convexity reports;
8. compute key-rate duration and key-rate DV01;
9. run curve-shock scenario analysis;
10. compare full revaluation with duration-convexity approximations;
11. compare barbell and bullet risk;
12. perform simple liability immunisation;
13. size a futures-style duration hedge;
14. test hedge performance under curve scenarios;
15. value the portfolio across a yield-curve history;
16. attribute returns to level, slope, and curvature;
17. estimate tail risk from curve-driven PnL;
18. create governance flags and audit checks;
19. save outputs and metadata.

The key computational takeaway:

> Fixed-income risk needs cashflow-level pricing and curve-level sensitivities, not just return covariance.

The key financial takeaway:

> Duration tells you first-order rate risk, convexity tells you curvature, and key-rate DV01 tells you where on the curve the risk actually lives.

## 31. Further reading

- Tuckman and Serrat, *Fixed Income Securities.*
- Fabozzi, *Bond Markets, Analysis, and Strategies.*
- Hull, *Options, Futures, and Other Derivatives.*
- Martellini, Priaulet, and Priaulet, *Fixed-Income Securities.*
- Litterman and Scheinkman on level, slope, and curvature in yield curves.
- Literature on key-rate duration, Treasury futures basis, liability-driven investing, and fixed-income risk management.